# 🕵️ Multi-Agent AI Code Review System

A portfolio-ready Python project that uses specialized reviewer agents to inspect code for likely bugs, security risks, maintainability issues, and missing tests. Findings are aggregated into a severity-ranked report.

> The core notebook uses Python's standard library and does not require a paid LLM API. Static checks are heuristic: always verify results with a human reviewer and real tests.


## 🏗️ Architecture

```text
Python Source
     ↓
Input Parser
     ├── 🐛 Correctness Agent
     ├── 🛡️ Security Agent
     ├── 🧼 Quality Agent
     └── 🧪 Test Agent
              ↓
       🧠 Orchestrator
              ↓
     🧾 Ranked JSON Report
              ↓
        👩‍💻 Human Review
```


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import ast
import json
import re
from collections import Counter

print("Environment ready ✅")


## 📥 1. Load sample code

This intentionally flawed sample helps demonstrate findings. Only review code you are authorized to inspect.

In [ ]:
sample_code = '''
import os
import subprocess

API_KEY = "example-hardcoded-secret"

def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + user_id
    result = eval(user_id)
    subprocess.run("echo " + user_id, shell=True)
    return query

def calculate_total(items):
    total = 0
    for item in items:
        total = total + item["price"]
    return total

def unused_helper(value):
    temp = value * 2
    return temp
'''
print(sample_code)


## 🧩 2. Finding schema

In [ ]:
@dataclass
class Finding:
    agent: str
    title: str
    severity: str
    evidence: str
    recommendation: str
    confidence: float = 0.75

SEVERITY_ORDER = {"critical": 0, "high": 1, "medium": 2, "low": 3, "info": 4}


## 🐛 3. Correctness agent

In [ ]:
def bug_review_agent(source: str) -> list[Finding]:
    findings = []
    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        return [Finding("bug-agent", "Python syntax error", "high",
                        f"{exc.msg} at line {exc.lineno}",
                        "Fix the syntax error and rerun parsing.", 0.99)]
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "eval":
            findings.append(Finding("bug-agent", "Dynamic evaluation used", "high",
                f"eval(...) at line {node.lineno}",
                "Replace eval with explicit parsing or a safe mapping.", 0.92))
        if isinstance(node, ast.FunctionDef) and not ast.get_docstring(node) and len(node.body) > 1:
            findings.append(Finding("bug-agent", "Function lacks a docstring", "low",
                f"{node.name}() at line {node.lineno}",
                "Add a concise docstring describing inputs, outputs, and errors.", 0.65))
    return findings

for item in bug_review_agent(sample_code):
    print(asdict(item))


## 🛡️ 4. Security agent

In [ ]:
def security_review_agent(source: str) -> list[Finding]:
    checks = [
        (r"\beval\s*\(", "Dynamic evaluation", "high",
         "Avoid eval on input; parse expected values safely."),
        (r"shell\s*=\s*True", "Shell execution enabled", "high",
         "Prefer argument lists with shell=False and validate arguments."),
        (r'API_KEY\s*=\s*[\'\"][^\'\"]+[\'\"]', "Possible hard-coded secret", "critical",
         "Move secrets to environment variables; rotate any real exposed key."),
        (r"SELECT\s+.*\+\s*\w+", "Possible SQL string concatenation", "high",
         "Use parameterized queries rather than concatenating user input."),
    ]
    findings = []
    for pattern, title, severity, recommendation in checks:
        match = re.search(pattern, source, re.IGNORECASE)
        if match:
            line = source[:match.start()].count("\n") + 1
            findings.append(Finding("security-agent", title, severity,
                f"Pattern near line {line}: {match.group(0)[:80]}",
                recommendation, 0.82))
    return findings

for item in security_review_agent(sample_code):
    print(asdict(item))


## 🧼 5. Code quality agent

In [ ]:
def quality_review_agent(source: str) -> list[Finding]:
    findings = []
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return findings
    imported = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            imported.extend(alias.name.split(".")[0] for alias in node.names)
        elif isinstance(node, ast.ImportFrom):
            imported.extend(alias.name for alias in node.names)
    for name in imported:
        if len(re.findall(rf"\b{re.escape(name)}\b", source)) <= 1:
            findings.append(Finding("quality-agent", "Potential unused import", "low",
                f"Import '{name}' appears only in its import statement.",
                "Remove if unused; confirm with a linter.", 0.68))
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name.startswith("unused"):
            findings.append(Finding("quality-agent", "Possibly unused helper", "low",
                f"Function '{node.name}' at line {node.lineno}.",
                "Confirm whether it is referenced; remove or document it.", 0.60))
    return findings

for item in quality_review_agent(sample_code):
    print(asdict(item))


## 🧪 6. Test agent

In [ ]:
def test_review_agent(source: str) -> list[Finding]:
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return []
    functions = [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
    tests = [n for n in functions if n.name.startswith("test_")]
    production = [n for n in functions if not n.name.startswith("test_")]
    if production and not tests:
        return [Finding("test-agent", "No tests found in supplied code", "medium",
            f"{len(production)} production function(s), 0 test function(s).",
            "Add tests for valid input, invalid input, edge cases, and failure paths.", 0.78)]
    return []

for item in test_review_agent(sample_code):
    print(asdict(item))


## 🧠 7. Orchestrator and report aggregation

In [ ]:
def run_code_review(source: str) -> list[Finding]:
    findings = []
    findings += bug_review_agent(source)
    findings += security_review_agent(source)
    findings += quality_review_agent(source)
    findings += test_review_agent(source)
    unique = {(f.agent, f.title, f.evidence): f for f in findings}
    return sorted(unique.values(),
                  key=lambda f: (SEVERITY_ORDER.get(f.severity, 99), -f.confidence))

findings = run_code_review(sample_code)
for i, f in enumerate(findings, 1):
    print(f"{i}. [{f.severity.upper()}] {f.title} — {f.agent}")
    print("   Evidence:", f.evidence)
    print("   Recommendation:", f.recommendation)


## 📊 8. Summary metrics and JSON export

In [ ]:
counts = Counter(f.severity for f in findings)
summary = {
    "files_reviewed": 1,
    "findings_total": len(findings),
    "by_severity": dict(counts),
    "agents_run": sorted(set(f.agent for f in findings)),
    "human_review_required": True,
}
report = {
    "summary": summary,
    "findings": [asdict(f) for f in findings],
    "disclaimer": "Heuristic findings may be false positives or miss issues.",
}
report_path = Path("code_review_report.json")
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("Saved:", report_path.resolve())


## 🤖 Upgrade to LLM-powered agents

Replace or augment deterministic checks with specialized LLM reviewers:

- 🐛 Correctness agent: logic flaws, edge cases, and error handling
- 🛡️ Security agent: vulnerability patterns and secure alternatives
- 🧼 Quality agent: complexity, duplication, naming, and maintainability
- 🧪 Testing agent: missing tests and suggested cases
- 🧠 Lead reviewer: deduplicate findings, resolve disagreement, and write the final report

Request structured JSON and validate it against the `Finding` schema. Keep API keys in environment variables; do not send confidential code or secrets to external services without authorization.


## 🔌 Production integrations

- 🐙 GitHub Pull Request API for diffs and review comments
- 🧪 pytest and CI checks
- 🔍 Ruff, Bandit, Semgrep, mypy, and dependency scanners
- 🧠 LangGraph for stateful orchestration
- 🔌 MCP for standardized repository tools
- 🌐 Streamlit or React dashboard
- 📈 LangSmith or custom traces for evaluation

## 🛡️ Safety and limitations

- Static checks are heuristics and may produce false positives or miss issues.
- Never execute untrusted code directly on the host.
- Run tests in a sandbox with timeouts and restricted permissions.
- Treat model output as untrusted input and validate structured results.
- Require human review before merging or deploying.
- Redact secrets before sending code to external services.


## 📏 Evaluation plan

Measure precision/recall against a labeled review set, severity agreement with experts, false-positive rate, latency, token cost, and reviewer acceptance.

## 🗺️ Roadmap

- [x] Deterministic specialist agents
- [x] Severity-ranked findings
- [x] JSON report export
- [ ] LLM-powered structured reviews
- [ ] LangGraph orchestration and retries
- [ ] GitHub PR diff integration
- [ ] Inline review comments
- [ ] Sandboxed test execution
- [ ] RAG over team coding standards
- [ ] Dashboard and evaluation suite

## 💼 Resume-ready description

> Built a multi-agent AI code review system with specialized correctness, security, code-quality, and test-coverage reviewers. Implemented severity-ranked findings, evidence-based recommendations, report aggregation, and JSON export, with a roadmap for LLM orchestration and GitHub pull-request integration.
